Conectamos al google colab para el guardado de los chaeckpoints el dataset y el zip en el drive


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR      = '/content/drive/MyDrive/NCA_Project'
ZIP_PATH       = f'{DRIVE_DIR}/celeba-dataset.zip'
IMG_DIR        = f'{DRIVE_DIR}/celeba/img_align_celeba/img_align_celeba'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'

print("ZIP existe:     ", os.path.exists(ZIP_PATH))
print("Imágenes:       ", len(os.listdir(IMG_DIR)) if os.path.exists(IMG_DIR) else "carpeta no existe")
print("Checkpoints:    ", os.path.exists(CHECKPOINT_DIR))

Mounted at /content/drive
ZIP existe:      True
Imágenes:        10000
Checkpoints:     True


In [ ]:
if not os.path.exists(ZIP_PATH):
    os.system(f'kaggle datasets download -d jessicali9530/celeba-dataset -p "{DRIVE_DIR}"')
    print("descargando")
else:
    print("ya esta descargado")

ya esta descargado


In [ ]:
!zip -r /content/NCA_Project.zip /content/drive/MyDrive/NCA_Project

Se truncaron las últimas líneas 5000 del resultado de transmisión.
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004006.jpg (deflated 3%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004007.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004008.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004009.jpg (deflated 3%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004010.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004011.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004012.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/img_align_celeba/img_align_celeba/004013.jpg (deflated 2%)
  adding: content/drive/MyDrive/NCA_Project/celeba/im

In [ ]:
if not os.path.exists(IMG_DIR) or len(os.listdir(IMG_DIR)) == 0:
    print("Extrayendo imágenes en Drive")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        img_files = [f for f in z.namelist() if f.endswith('.jpg')][:10000]
        for f in img_files:
            z.extract(f, f'{DRIVE_DIR}/celeba/')
    print(f"{len(img_files)} imágenes extraídas en Drive")
else:
    print(f"Ya hay {len(os.listdir(IMG_DIR))} imágenes en Drive")

Ya hay 10000 imágenes en Drive


In [ ]:
!ls '/content/drive/MyDrive/NCA_Project'

celeba	celeba-dataset.zip  checkpoints  reconstruction.mp4


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PerceptionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        lap     = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]],   dtype=torch.float32)
        for name, k in [('sx', sobel_x), ('sy', sobel_y), ('lap', lap)]:
            self.register_buffer(name, k.view(1,1,3,3).repeat(channels,1,1,1))

    def forward(self, x):
        gx  = F.conv2d(x, self.sx,  padding=1, groups=self.channels)
        gy  = F.conv2d(x, self.sy,  padding=1, groups=self.channels)
        lap = F.conv2d(x, self.lap, padding=1, groups=self.channels)
        return torch.cat([x, gx, gy, lap], dim=1)


class UpdateNetwork(nn.Module):
    def __init__(self, channels, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels * 4, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden // 2, 1),
            nn.GELU(),
            nn.Conv2d(hidden // 2, channels, 1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class NCA(nn.Module):
    def __init__(self, channels=96, update_rate=0.5):
        super().__init__()
        self.channels    = channels
        self.update_rate = update_rate
        self.perceive    = PerceptionBlock(channels)
        self.update      = UpdateNetwork(channels)

    def forward(self, x):
        perceived = self.perceive(x)
        dx        = self.update(perceived)
        alive     = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3],
                                device=x.device) < self.update_rate).float()
        return x + dx * alive

    @staticmethod
    def init_state(img, mask, channels=96):
        B, _, H, W = img.shape
        state         = torch.zeros(B, channels, H, W, device=img.device)
        state[:, :3]  = img
        state[:, 3:4] = mask
        state[:, 4:7] = img * mask
        return state

print('✓ NCA definida')

✓ NCA definida


In [ ]:
import torchvision.models as models
from torchvision.transforms import Normalize


def gram_matrix(feat):
    B, C, H, W = feat.shape
    f = feat.view(B, C, H * W)
    g = torch.bmm(f, f.transpose(1, 2)) / (C * H * W)
    return g.clamp(-1e3, 1e3)


class VGGFeatures(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg          = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features.eval().to(device)
        self.slice1  = vgg[:4]
        self.slice2  = vgg[4:9]
        self.slice3  = vgg[9:16]
        for p in self.parameters():
            p.requires_grad = False
        self.norm = Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])

    def forward(self, x):
        x  = self.norm(x.clamp(0, 1))
        f1 = self.slice1(x)
        f2 = self.slice2(f1)
        f3 = self.slice3(f2)
        return f1, f2, f3


class TotalLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.vgg = VGGFeatures(device)

    def forward(self, output, target, mask):
        hole  = 1.0 - mask
        diff  = (output - target).abs()

        hole_l1  = (diff * hole).sum()  / (hole.sum()  * 3 + 1e-8)
        valid_l1 = (diff * mask).sum()  / (mask.sum()  * 3 + 1e-8)

        of1, of2, of3 = self.vgg(output)
        tf1, tf2, tf3 = self.vgg(target)

        perc  = (F.l1_loss(of1, tf1) + F.l1_loss(of2, tf2) + F.l1_loss(of3, tf3))
        style = (F.l1_loss(gram_matrix(of1), gram_matrix(tf1)) +
                 F.l1_loss(gram_matrix(of2), gram_matrix(tf2)) +
                 F.l1_loss(gram_matrix(of3), gram_matrix(tf3)))
        tv    = (torch.mean(torch.abs(output[:,:,:,:-1] - output[:,:,:,1:])) +
                 torch.mean(torch.abs(output[:,:,:-1,:] - output[:,:,1:,:])))

        total = hole_l1 + 0.5*valid_l1 + 0.05*perc + 0.05*style + 0.005*tv


        if torch.isnan(total) or torch.isinf(total):
            return hole_l1 + 0.5*valid_l1, {'hole': hole_l1.item(), 'valid': valid_l1.item(),
                                              'perc': 0.0, 'style': 0.0, 'tv': tv.item()}

        return total, {'hole': hole_l1.item(), 'perc': perc.item(),
                       'style': style.item(), 'tv': tv.item()}

print('✓ Loss definida')

✓ Loss definida


In [ ]:
import random
import math


def _draw_stroke(mask, W, H):
    x, y = random.randint(0, W), random.randint(0, H)
    for _ in range(random.randint(15, 40)):
        angle  = random.uniform(0, 2 * math.pi)
        length = random.randint(5, 15)
        r      = random.randint(2, 10)
        x = max(0, min(W-1, int(x + length * math.cos(angle))))
        y = max(0, min(H-1, int(y + length * math.sin(angle))))
        mask[max(0,y-r):min(H,y+r+1), max(0,x-r):min(W,x+r+1)] = 0


def _draw_box(mask, W, H):
    bw = random.randint(8, 28)
    bh = random.randint(8, 28)
    x  = random.randint(0, max(1, W - bw))
    y  = random.randint(0, max(1, H - bh))
    mask[y:y+bh, x:x+bw] = 0


def random_mask_batch(imgs):
    B, C, H, W = imgs.shape
    masks = torch.ones(B, 1, H, W, device=imgs.device)
    for i in range(B):
        m    = masks[i, 0].cpu()
        mode = random.choice(['boxes', 'strokes', 'mixed'])
        if mode == 'boxes':
            for _ in range(random.randint(1, 5)):
                _draw_box(m, W, H)
        elif mode == 'strokes':
            for _ in range(random.randint(1, 4)):
                _draw_stroke(m, W, H)
        else:
            _draw_box(m, W, H)
            _draw_stroke(m, W, H)
        masks[i, 0] = m.to(imgs.device)
    return imgs * masks, masks

print('✓ Masking definido')

✓ Masking definido


In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class CelebADataset(Dataset):
    def __init__(self, folder, size=100, max_imgs=8000):
        files       = [f for f in os.listdir(folder)
                       if f.lower().endswith(('.jpg','.png','.jpeg'))]
        self.files  = files[:max_imgs]
        self.folder = folder
        self.tf     = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
        ])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.folder, self.files[idx])).convert('RGB')
        return self.tf(img)

print('✓ Dataset definido')

✓ Dataset definido


In [ ]:
import torch, os, random, gc
from torch.utils.data import DataLoader

DRIVE_DIR      = '/content/drive/MyDrive/NCA_Project'
DATASET_PATH   = f'{DRIVE_DIR}/celeba/img_align_celeba/img_align_celeba'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
CKPT_LATEST    = f'{CHECKPOINT_DIR}/latest.pth'
CKPT_BEST      = f'{CHECKPOINT_DIR}/best.pth'

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
CHANNELS    = 32
BATCH_SIZE  = 6
ACCUM_STEPS = 2
EPOCHS      = 300
SAVE_EVERY  = 300
WARM_STEPS  = 20  # pasos sin gradiente
TRAIN_STEPS = 8   # pasos con gradiente

dataset = CelebADataset(DATASET_PATH)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     drop_last=True, num_workers=2, pin_memory=True)

model     = NCA(channels=CHANNELS).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4, steps_per_epoch=len(loader), epochs=EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
loss_fn   = TotalLoss(DEVICE)

start_epoch = 0
global_step = 0
best_loss   = float('inf')
nan_skiped = 0

if os.path.exists(CKPT_LATEST):
    ckpt        = torch.load(CKPT_LATEST, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    global_step = ckpt['step']
    best_loss   = ckpt.get('best_loss', best_loss)
    print(f'✓ Retomando epoch {start_epoch}, step {global_step}')

print(f'Entrenando en {DEVICE} | {len(dataset)} imágenes | batch {BATCH_SIZE} | channels {CHANNELS}')

for epoch in range(start_epoch, EPOCHS):
    epoch_loss = 0
    epoch_steps = 0
    optimizer.zero_grad()

    for i, imgs in enumerate(loader):
        imgs              = imgs.to(DEVICE,non_blocking=True)
        corrupted, mask_t = random_mask_batch(imgs)
        state             = NCA.init_state(corrupted, mask_t, channels=CHANNELS)

        # Calentamiento sin gradientes (no acumula memoria)
        with torch.no_grad():
            for _ in range(WARM_STEPS):
                state        = model(state)
                state[:, :3] = state[:, :3] * (1 - mask_t) + corrupted * mask_t

        # Solo backprop en los últimos pasos
        state = state.detach()
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            for _ in range(TRAIN_STEPS):
                state        = model(state)
                state[:, :3] = state[:, :3] * (1 - mask_t) + corrupted * mask_t
            output        = torch.clamp(state[:, :3], 0, 1)
            loss, details = loss_fn(output, imgs, mask_t)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_skipped += 1
            optimizer.zero_grad()
            del state, output, loss, corrupted, mask_t
            torch.cuda.empty_cache()
            continue

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()
            scheduler.step()

            epoch_loss  += loss.item()
            global_step += 1
            epoch_steps += 1

        if global_step % 20 == 0:
            free = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"Ep {epoch:03d} | Step {global_step} | "
                  f"Loss {loss.item():.4f} | "
                  f"hole {details['hole']:.3f} | "
                  f"perc {details['perc']:.3f} | "
                  f"style {details['style']:.4f} | "
                  f"VRAM libre {free:.2f}GB")

        if global_step % SAVE_EVERY == 0:
            torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict(), 'epoch': epoch,
                        'step': global_step, 'best_loss': best_loss}, CKPT_LATEST)
            print(f'✓ Checkpoint guardado step {global_step}')

        del state, output, loss, corrupted, mask_t
        gc.collect()
        torch.cuda.empty_cache()

    avg = epoch_loss / len(loader)
    print(f'── Epoch {epoch} | Avg Loss: {avg:.4f}')

    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), CKPT_BEST)
        print(f'✓ Mejor modelo guardado (loss {best_loss:.4f})')

/tmp/ipykernel_7908/1786223917.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))


KeyboardInterrupt: 

In [ ]:
import cv2, os, random
import numpy as np
from PIL import Image

DATASET_PATH   = f'{DRIVE_DIR}/celeba/img_align_celeba/img_align_celeba'

def load_image(path, size=100):
    img = Image.open(path).convert('RGB').resize((size, size))
    img = torch.tensor(np.array(img) / 255.0).permute(2,0,1).float()
    return img.unsqueeze(0).to(DEVICE)

test_file = random.choice(os.listdir(DATASET_PATH))
TEST_IMG  = os.path.join(DATASET_PATH, test_file)
OUT_VIDEO = f'{DRIVE_DIR}/reconstruction.mp4'
print(f'Usando: {test_file}')

model_inf = NCA(channels=CHANNELS).to(DEVICE)
ckpt      = torch.load(CKPT_BEST if os.path.exists(CKPT_BEST) else CKPT_LATEST, map_location=DEVICE)
model_inf.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
model_inf.eval()
print('✓ Modelo cargado')

img               = load_image(TEST_IMG)
corrupted, mask_t = random_mask_batch(img)
state             = NCA.init_state(corrupted, mask_t, channels=CHANNELS)

frames = []
with torch.no_grad():
    for step in range(120):
        state        = model_inf(state)
        state[:, :3] = state[:, :3] * (1 - mask_t) + corrupted * mask_t
        output       = torch.clamp(state[:, :3], 0, 1)

        # Verificar que no haya NaN antes de agregar frame
        if torch.isnan(output).any():
            output = corrupted  # si hay NaN usa la imagen corrupta
            print(f'⚠️ NaN en frame {step}, usando corrupted')

        frame = (output[0].permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        frame = cv2.resize(frame, (400, 400), interpolation=cv2.INTER_LANCZOS4)
        frames.append(frame)

h, w  = frames[0].shape[:2]
video = cv2.VideoWriter(OUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
for f in frames: video.write(f)
video.release()
print(f'✓ Video guardado en {OUT_VIDEO}')

Usando: 001795.jpg
✓ Modelo cargado
✓ Video guardado en /content/drive/MyDrive/NCA_Project/reconstruction.mp4
